# SemEval-2026 Task 13 — Subtask B: GraphCodeBERT (Improved)

**Multi-Class Authorship Detection** — 11 classes (human + 10 LLM families)

### Model: `microsoft/graphcodebert-base`
- Pre-trained with data flow information → structure-aware representations
- Same RoBERTa architecture as CodeBERT → drop-in replacement

### Improvements over baseline (macro F1 0.43 → target ≥ 0.65)
- **Focal Loss** (γ=2) + **SupConLoss** auxiliary head (weight 0.2)
- **Aggressive class weighting** via sklearn balanced + clip@8
- **max_length=512** with fp16 enabled (T4 supports it)
- **Layer-wise LR decay** (LLRD, decay=0.9)
- **Embedding-space Mixup** (Beta(0.4,0.4))
- **Label smoothing removed** (0.0)
- **Per-class threshold calibration** on val set
- **Top-3 checkpoint averaging** by macro F1

### Setup
- **GPU:** Kaggle Tesla T4 (16 GB VRAM)
- **Data:** 10% effective via human class capping
- **Metric:** Macro F1 (competition metric)

Set runtime to `GPU T4 x2` or `GPU T4` in Kaggle settings.

In [1]:
# ============================================================
# Cell 1: Environment Setup  # CHANGED — added scipy
# ============================================================
# NOTE: torch==2.1.2 not available for cu118. Using pre-installed PyTorch.
!pip install --upgrade transformers==4.45.0 huggingface_hub datasets scikit-learn accelerate scipy -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 73.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 89.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 53.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 91.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydata-profiling 4.18.1 requires scipy<1.17,>=1.8, but you have scipy 1.17.1 which is incompatible.


## Data Setup
Uncomment the option matching your environment.

In [2]:
# ============================================================
# Cell 2: Data paths
# ============================================================

# --- OPTION A: Kaggle input paths ---
# TRAIN_PATH = "/kaggle/input/datasets/daniilor/semeval-2026-task13/SemEval-2026-Task13/task_b/task_b_training_set.parquet"
# VAL_PATH   = "/kaggle/input/datasets/daniilor/semeval-2026-task13/SemEval-2026-Task13/task_b/task_b_validation_set.parquet"
# TEST_PATH  = "/kaggle/input/datasets/daniilor/semeval-2026-task13/SemEval-2026-Task13/task_b/task_b_test_set_sample.parquet"

# --- OPTION B: Download from HuggingFace (default) ---
from datasets import load_dataset
print("Downloading SemEval-2026-Task13 Subtask B from HuggingFace...")
hf_dataset = load_dataset("DaniilOr/SemEval-2026-Task13", "B")
hf_dataset['train'].to_parquet('task_b_train.parquet')
hf_dataset['validation'].to_parquet('task_b_val.parquet')
if 'test' in hf_dataset:
    hf_dataset['test'].to_parquet('task_b_test.parquet')
    TEST_PATH = 'task_b_test.parquet'
else:
    TEST_PATH = None
    print("No test split available on HuggingFace.")
TRAIN_PATH = 'task_b_train.parquet'
VAL_PATH   = 'task_b_val.parquet'
print("Done!")

print(f"Train: {TRAIN_PATH}")
print(f"Val:   {VAL_PATH}")
print(f"Test:  {TEST_PATH}")


README.md:   0%|          | 0.00/801 [00:00<?, ?B/s]

task_b/task_b_training_set.parquet:   0%|          | 0.00/296M [00:00<?, ?B/s]

task_b/task_b_validation_set.parquet:   0%|          | 0.00/59.3M [00:00<?, ?B/s]

task_b/task_b_test_set_sample.parquet:   0%|          | 0.00/666k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/500000 [00:00<?, ? examples/s]

...

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/8 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Done!
Train: task_b_train.parquet
Val:   task_b_val.parquet
Test:  task_b_test.parquet


## Configuration

In [3]:
# ============================================================
# Cell 3: Configuration  # CHANGED — many values updated
# ============================================================

# --- Model ---
MODEL_NAME = "microsoft/graphcodebert-base"

# --- Subsampling ---
USE_SUBSET        = True
HUMAN_SUBSET_SIZE = 45000
VAL_FRACTION      = 0.1
RANDOM_SEED       = 42

# --- Training hyperparameters ---
MAX_LENGTH         = 512      # CHANGED from 256 — capture longer AI patterns
BATCH_SIZE         = 8        # CHANGED from 16 — fit max_length=512 on T4
GRAD_ACCUM_STEPS   = 4        # CHANGED from 2 — effective batch = 32
LEARNING_RATE      = 3e-5     # CHANGED from 2e-5
NUM_EPOCHS         = 6        # CHANGED from 10
WEIGHT_DECAY       = 0.01
WARMUP_STEPS       = 500      # CHANGED from ratio — use fixed steps
LABEL_SMOOTHING    = 0.0      # CHANGED from 0.1 — hurts minority classes
EARLY_STOPPING_PATIENCE = 2   # CHANGED from 3

# --- CHANGED: Focal Loss + SupCon config ---
FOCAL_GAMMA        = 2.0
SUPCON_WEIGHT      = 0.2
SUPCON_TEMPERATURE = 0.07

# --- CHANGED: LLRD config ---
LLRD_DECAY         = 0.9

# --- CHANGED: Mixup config ---
MIXUP_ALPHA        = 0.4

# --- CHANGED: Class weight config ---
MAX_CLASS_WEIGHT   = 8.0

# --- Output ---
OUTPUT_DIR = "/kaggle/working/results_graphcodebert_taskB"

# --- Label mapping ---
ID_TO_LABEL = {
    0: "human", 1: "deepseek", 2: "qwen", 3: "01-ai", 4: "bigcode",
    5: "gemma", 6: "phi", 7: "meta-llama", 8: "ibm-granite", 9: "mistral", 10: "openai"
}
LABEL_TO_ID = {v: k for k, v in ID_TO_LABEL.items()}
NUM_LABELS = len(ID_TO_LABEL)

print(f"Model: {MODEL_NAME}")
print(f"Task B: {NUM_LABELS}-class classification")
print(f"Labels: {list(ID_TO_LABEL.values())}")
print(f"Subset mode: {USE_SUBSET}")
print(f"Human cap: {HUMAN_SUBSET_SIZE}")
print(f"Max length: {MAX_LENGTH}")
print(f"Label smoothing: {LABEL_SMOOTHING}")
print(f"Focal gamma: {FOCAL_GAMMA}")
print(f"SupCon weight: {SUPCON_WEIGHT}")
print(f"LLRD decay: {LLRD_DECAY}")
print(f"Mixup alpha: {MIXUP_ALPHA}")


Model: microsoft/graphcodebert-base
Task B: 11-class classification
Labels: ['human', 'deepseek', 'qwen', '01-ai', 'bigcode', 'gemma', 'phi', 'meta-llama', 'ibm-granite', 'mistral', 'openai']
Subset mode: True
Human cap: 45000
Max length: 512
Label smoothing: 0.0
Focal gamma: 2.0
SupCon weight: 0.2
LLRD decay: 0.9
Mixup alpha: 0.4


In [4]:
# ============================================================
# Cell 4: Imports  # CHANGED — added scipy, copy, collections
# ============================================================

import os
os.environ["WANDB_DISABLED"] = "true"

import gc, copy, random
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from collections import Counter
from datasets import Dataset
import transformers
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    RobertaModel,
    RobertaPreTrainedModel,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding,
)
from transformers.modeling_outputs import SequenceClassifierOutput
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    f1_score,
    confusion_matrix,
)
from sklearn.utils.class_weight import compute_class_weight  # CHANGED
from scipy.optimize import minimize_scalar  # CHANGED — for threshold calibration
import warnings
warnings.filterwarnings("ignore")

print(f"PyTorch version: {torch.__version__}")
print(f"Transformers version: {transformers.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    vram = getattr(props, 'total_memory', getattr(props, 'total_mem', 0))
    print(f"VRAM: {vram / 1e9:.1f} GB")


The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

2026-04-05 21:58:30.336971: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775426310.522406      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775426310.580012      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775426311.056881      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775426311.056908      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775426311.056911      23 computation_placer.cc:177] computation placer alr

PyTorch version: 2.10.0+cu128
Transformers version: 4.45.0
CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


## Loss Functions, Model & Utilities

- **FocalLoss**: suppresses easy 'human' examples, focuses on hard minorities
- **SupConLoss**: contrastive loss on [CLS] embeddings for inter-class separation
- **Balanced class weights**: sklearn compute_class_weight('balanced'), clipped at 8
- **LLRD**: layer-wise learning rate decay for stable fine-tuning
- **Mixup**: embedding-space interpolation for minority augmentation

In [5]:
# ============================================================
# Cell 5: FocalLoss, SupConLoss, Model, Trainer, Utilities
# CHANGED — completely rewritten
# ============================================================

# ── Focal Loss ──────────────────────────────────────────────
class FocalLoss(nn.Module):
    """Focal Loss with per-class alpha weights and gamma focusing."""
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super().__init__()
        self.gamma = gamma
        self.reduction = reduction
        if alpha is not None:
            self.register_buffer('alpha', alpha.float())
        else:
            self.alpha = None

    def forward(self, logits, targets):
        ce_loss = F.cross_entropy(logits, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_weight = (1 - pt) ** self.gamma
        if self.alpha is not None:
            alpha_t = self.alpha.to(logits.device)[targets]
            focal_weight = alpha_t * focal_weight
        loss = focal_weight * ce_loss
        if self.reduction == 'mean':
            return loss.mean()
        return loss.sum()


# ── Supervised Contrastive Loss ─────────────────────────────
class SupConLoss(nn.Module):
    """Supervised contrastive loss on normalized embeddings."""
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature

    def forward(self, features, labels):
        # features: (B, D), labels: (B,)
        features = F.normalize(features, dim=1)
        B = features.shape[0]
        if B < 2:
            return torch.tensor(0.0, device=features.device)
        sim = torch.matmul(features, features.T) / self.temperature  # (B, B)
        # mask: same-class pairs
        labels = labels.unsqueeze(1)
        mask = (labels == labels.T).float()  # (B, B)
        # remove self-similarity
        logits_mask = 1.0 - torch.eye(B, device=features.device)
        mask = mask * logits_mask
        # log-sum-exp
        exp_sim = torch.exp(sim) * logits_mask
        log_prob = sim - torch.log(exp_sim.sum(dim=1, keepdim=True) + 1e-8)
        # mean of log-likelihood over positive pairs
        pos_count = mask.sum(dim=1)
        mean_log_prob = (mask * log_prob).sum(dim=1) / (pos_count + 1e-8)
        # only consider anchors with at least one positive
        valid = pos_count > 0
        if valid.sum() == 0:
            return torch.tensor(0.0, device=features.device)
        loss = -mean_log_prob[valid].mean()
        return loss


# ── Custom Model with projection head ──────────────────────
class RobertaForClassificationWithSupCon(RobertaPreTrainedModel):
    """RoBERTa + classification head + contrastive projection head."""
    def __init__(self, config):
        super().__init__(config)
        self.num_labels = config.num_labels
        self.roberta = RobertaModel(config, add_pooling_layer=False)
        hidden = config.hidden_size
        # Classification head
        self.classifier = nn.Sequential(
            nn.Dropout(config.hidden_dropout_prob),
            nn.Linear(hidden, hidden),
            nn.Tanh(),
            nn.Dropout(config.hidden_dropout_prob),
            nn.Linear(hidden, config.num_labels),
        )
        # Projection head for SupCon
        self.projector = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 128),
        )
        self.post_init()

    def forward(self, input_ids=None, attention_mask=None, labels=None, **kw):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        cls_emb = outputs.last_hidden_state[:, 0, :]  # [CLS]
        logits = self.classifier(cls_emb)
        proj = self.projector(cls_emb)
        # Store proj for the trainer to use
        return SequenceClassifierOutput(
            logits=logits,
            hidden_states=(proj,),  # pass projection via hidden_states
        )


# ── Utilities ───────────────────────────────────────────────
def stratified_subsample(df, human_subset_size=HUMAN_SUBSET_SIZE,
                         random_state=RANDOM_SEED):
    """Keep ALL minority samples, downsample human class."""
    human_df = df[df['label'] == 0]
    minority_df = df[df['label'] != 0]
    original_total = len(df)
    if len(human_df) > human_subset_size:
        human_df = human_df.sample(n=human_subset_size, random_state=random_state)
    result = pd.concat([human_df, minority_df], ignore_index=True)
    result = result.sample(frac=1, random_state=random_state).reset_index(drop=True)
    pct = len(result) / original_total * 100
    print(f"  Subsampled: {original_total} -> {len(result)} samples ({pct:.1f}%)")
    print(f"  Human: {len(human_df)} | Minority (all 10 classes): {len(minority_df)}")
    return result


def compute_balanced_weights(labels, num_labels, max_weight=MAX_CLASS_WEIGHT):  # CHANGED
    """sklearn balanced class weights, clipped at max_weight."""
    classes = np.arange(num_labels)
    weights = compute_class_weight('balanced', classes=classes, y=labels)
    weights = np.clip(weights, 1.0, max_weight)
    return torch.FloatTensor(weights)


# ── LLRD optimizer ──────────────────────────────────────────
def get_llrd_optimizer(model, base_lr=LEARNING_RATE, decay=LLRD_DECAY,
                       weight_decay=WEIGHT_DECAY):  # CHANGED — new function
    """Layer-wise learning rate decay for RoBERTa."""
    opt_params = []
    no_decay = ['bias', 'LayerNorm.weight', 'LayerNorm.bias']

    # Encoder layers
    num_layers = model.config.num_hidden_layers  # 12 for base
    for layer_idx in range(num_layers):
        lr = base_lr * (decay ** (num_layers - layer_idx))
        layer_params = []
        layer_params_no_decay = []
        layer_name = f"roberta.encoder.layer.{layer_idx}."
        for n, p in model.named_parameters():
            if layer_name in n:
                if any(nd in n for nd in no_decay):
                    layer_params_no_decay.append(p)
                else:
                    layer_params.append(p)
        opt_params.append({'params': layer_params, 'lr': lr, 'weight_decay': weight_decay})
        opt_params.append({'params': layer_params_no_decay, 'lr': lr, 'weight_decay': 0.0})

    # Embeddings (lowest LR)
    embed_lr = base_lr * (decay ** (num_layers + 1))
    embed_params = [p for n, p in model.named_parameters()
                    if 'roberta.embeddings' in n]
    opt_params.append({'params': embed_params, 'lr': embed_lr, 'weight_decay': weight_decay})

    # Classifier + projector heads (highest LR)
    head_params = [p for n, p in model.named_parameters()
                   if 'classifier' in n or 'projector' in n]
    opt_params.append({'params': head_params, 'lr': base_lr, 'weight_decay': weight_decay})

    return torch.optim.AdamW(opt_params)


# ── Mixup helper ────────────────────────────────────────────
def mixup_data(embeddings, labels, alpha=MIXUP_ALPHA):  # CHANGED — new function
    """Embedding-space mixup with Beta distribution."""
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1.0
    batch_size = embeddings.size(0)
    index = torch.randperm(batch_size, device=embeddings.device)
    mixed_emb = lam * embeddings + (1 - lam) * embeddings[index]
    return mixed_emb, labels, labels[index], lam


# ── WeightedTrainer with Focal + SupCon + Mixup ────────────
class ImprovedTrainer(Trainer):  # CHANGED — renamed, completely rewritten
    """Custom Trainer: FocalLoss + SupConLoss + Mixup."""

    def __init__(self, class_weights=None, focal_gamma=FOCAL_GAMMA,
                 supcon_weight=SUPCON_WEIGHT, mixup_alpha=MIXUP_ALPHA, **kwargs):
        super().__init__(**kwargs)
        alpha = class_weights if class_weights is not None else None
        self.focal_loss = FocalLoss(alpha=alpha, gamma=focal_gamma)
        self.supcon_loss = SupConLoss(temperature=SUPCON_TEMPERATURE)
        self.supcon_weight = supcon_weight
        self.mixup_alpha = mixup_alpha

    def create_optimizer(self):  # CHANGED — use LLRD
        self.optimizer = get_llrd_optimizer(self.model)
        return self.optimizer

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        proj = outputs.hidden_states[0] if outputs.hidden_states else None

        # Focal loss (primary)
        loss_ce = self.focal_loss(logits, labels)

        # SupCon loss (auxiliary)
        loss_supcon = torch.tensor(0.0, device=logits.device)
        if proj is not None and self.supcon_weight > 0:
            loss_supcon = self.supcon_loss(proj, labels)

        loss = (1 - self.supcon_weight) * loss_ce + self.supcon_weight * loss_supcon
        return (loss, outputs) if return_outputs else loss


print("FocalLoss, SupConLoss, Model, ImprovedTrainer, LLRD, Mixup defined.")


FocalLoss, SupConLoss, Model, ImprovedTrainer, LLRD, Mixup defined.


## GraphCodeBERT Trainer for Task B

Improved pipeline with:
- FocalLoss + SupConLoss
- LLRD optimizer
- max_length=512, fp16=True
- Top-3 checkpoint averaging
- Per-class threshold calibration

In [6]:
# ============================================================
# Cell 6: GraphCodeBERT Trainer for Task B  # CHANGED — major rework
# ============================================================

class GraphCodeBERTTrainerB:
    """Task B trainer — improved pipeline."""

    def __init__(self, model_name=MODEL_NAME, max_length=MAX_LENGTH):
        self.model_name = model_name
        self.max_length = max_length
        self.tokenizer = None
        self.model = None
        self.class_weights = None
        self.tag = model_name.split("/")[-1]
        self.best_checkpoints = []  # CHANGED — track top-3 checkpoints

    # ── Data loading ────────────────────────────────────────
    def load_and_prepare_data(self):
        try:
            df = pd.read_parquet(TRAIN_PATH)
            print(f"[{self.tag}] Dataset columns: {df.columns.tolist()}")
            print(f"[{self.tag}] Original dataset size: {len(df)}")

            if 'code' not in df.columns or 'label' not in df.columns:
                raise ValueError("Dataset must contain 'code' and 'label' columns")

            df = df.dropna(subset=['code', 'label'])
            df['label'] = df['label'].astype(int)

            unique_labels = sorted(df['label'].unique())
            print(f"\n[{self.tag}] Unique labels: {unique_labels}")

            print(f"\n[{self.tag}] Full dataset label distribution:")
            label_counts = df['label'].value_counts().sort_index()
            for label_id, count in label_counts.items():
                name = ID_TO_LABEL.get(label_id, f"unknown-{label_id}")
                print(f"  {label_id} ({name:>12s}): {count:>7d} ({count/len(df)*100:.1f}%)")

            if USE_SUBSET:
                print(f"\n--- SUBSET MODE (target: 10% of dataset) ---")
                df = stratified_subsample(df, human_subset_size=HUMAN_SUBSET_SIZE)

            # CHANGED — use sklearn balanced weights clipped at 8
            self.class_weights = compute_balanced_weights(
                df['label'].values, NUM_LABELS, max_weight=MAX_CLASS_WEIGHT
            )

            print(f"\n[{self.tag}] Class weights (balanced, clipped@{MAX_CLASS_WEIGHT}):")
            label_counts = df['label'].value_counts().sort_index()
            for i, w in enumerate(self.class_weights.tolist()):
                name = ID_TO_LABEL.get(i, f"unknown-{i}")
                count = label_counts.get(i, 0)
                print(f"  {name:>12s}: weight={w:>5.2f}  (n={count})")

            val_df = pd.read_parquet(VAL_PATH)
            val_df['label'] = val_df['label'].astype(int)

            if USE_SUBSET:
                print(f"\n--- Subsampling validation to {VAL_FRACTION*100:.0f}% ---")
                val_df = val_df.groupby('label', group_keys=False).apply(
                    lambda x: x.sample(frac=VAL_FRACTION, random_state=RANDOM_SEED)
                ).reset_index(drop=True)

            print(f"\n[{self.tag}] Final -> Train: {len(df)} | Val: {len(val_df)}")
            return df, val_df

        except Exception as e:
            print(f"[{self.tag}] Error loading dataset: {e}")
            raise

    # ── Model + tokenizer ──────────────────────────────────
    def initialize_model_and_tokenizer(self):  # CHANGED — use custom model
        print(f"\n[{self.tag}] Initialising model and tokenizer ...")
        self.tokenizer = RobertaTokenizer.from_pretrained(self.model_name)

        # CHANGED — use custom model with SupCon projection head
        from transformers import RobertaConfig
        config = RobertaConfig.from_pretrained(self.model_name, num_labels=NUM_LABELS)
        self.model = RobertaForClassificationWithSupCon.from_pretrained(
            self.model_name, config=config,
        ).to('cuda')

        total_params = sum(p.numel() for p in self.model.parameters())
        train_params = sum(p.numel() for p in self.model.parameters() if p.requires_grad)
        print(f"[{self.tag}] {NUM_LABELS} labels | params {total_params:,} | trainable {train_params:,}")

    # ── Tokenization ───────────────────────────────────────
    def tokenize_function(self, examples):
        return self.tokenizer(
            examples['code'],
            truncation=True,
            padding=True,
            max_length=self.max_length,
            return_tensors="pt",
        )

    # ── Dataset preparation ────────────────────────────────
    def prepare_datasets(self, train_df, val_df):
        print(f"\n[{self.tag}] Tokenizing datasets...")
        train_dataset = Dataset.from_pandas(train_df[['code', 'label']])
        val_dataset   = Dataset.from_pandas(val_df[['code', 'label']])

        train_dataset = train_dataset.map(
            self.tokenize_function, batched=True, remove_columns=['code']
        )
        val_dataset = val_dataset.map(
            self.tokenize_function, batched=True, remove_columns=['code']
        )

        train_dataset = train_dataset.rename_column('label', 'labels')
        val_dataset   = val_dataset.rename_column('label', 'labels')

        print(f"[{self.tag}] Done. Train: {len(train_dataset)}, Val: {len(val_dataset)}")
        return train_dataset, val_dataset

    # ── Metrics ────────────────────────────────────────────
    def compute_metrics(self, eval_pred):
        predictions, labels = eval_pred
        preds = np.argmax(predictions, axis=1)

        unique, counts = np.unique(preds, return_counts=True)
        pred_dist = {int(u): int(c) for u, c in zip(unique, counts)}
        print(f"  Prediction distribution: {pred_dist}")

        accuracy = accuracy_score(labels, preds)
        macro_f1 = f1_score(labels, preds, average='macro', zero_division=0)
        weighted_f1 = f1_score(labels, preds, average='weighted', zero_division=0)

        # CHANGED — print per-class F1 at every eval step
        _, _, f1_per_class, _ = precision_recall_fscore_support(
            labels, preds, average=None, zero_division=0, labels=range(NUM_LABELS)
        )
        class_f1_str = ' | '.join(
            f"{ID_TO_LABEL[i][:3]}={f1_per_class[i]:.2f}" for i in range(NUM_LABELS)
        )
        print(f"  Per-class F1: {class_f1_str}")

        return {
            'macro_f1': macro_f1,
            'weighted_f1': weighted_f1,
            'accuracy': accuracy,
        }

    # ── Training ───────────────────────────────────────────
    def train(
        self,
        train_dataset,
        val_dataset,
        output_dir=OUTPUT_DIR,
        num_epochs=NUM_EPOCHS,
        batch_size=BATCH_SIZE,
        learning_rate=LEARNING_RATE,
    ):
        print(f"\n{'='*60}")
        print(f"  STARTING TRAINING — {self.tag}")
        print(f"{'='*60}")

        steps_per_epoch = len(train_dataset) // (batch_size * GRAD_ACCUM_STEPS)
        eval_save_steps = max(200, steps_per_epoch // 3)

        training_args = TrainingArguments(  # CHANGED — many updates
            output_dir=output_dir,
            num_train_epochs=num_epochs,
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size * 2,
            gradient_accumulation_steps=GRAD_ACCUM_STEPS,
            warmup_steps=WARMUP_STEPS,          # CHANGED from warmup_ratio
            weight_decay=WEIGHT_DECAY,
            logging_dir=f"{output_dir}/logs",
            logging_steps=50,
            eval_strategy="steps",
            eval_steps=eval_save_steps,
            save_strategy="steps",
            save_steps=eval_save_steps,
            load_best_model_at_end=True,
            metric_for_best_model="macro_f1",   # CHANGED — save on macro F1
            greater_is_better=True,
            remove_unused_columns=False,
            learning_rate=learning_rate,
            lr_scheduler_type="cosine",
            save_total_limit=5,                 # CHANGED from 2 — keep top checkpoints
            fp16=True,                          # CHANGED from False — T4 supports fp16
            report_to="none",
        )

        data_collator = DataCollatorWithPadding(tokenizer=self.tokenizer)

        # CHANGED — strip hidden_states during eval so compute_metrics sees only logits
        def _preprocess_logits_for_metrics(logits, labels):
            if isinstance(logits, tuple):
                return logits[0]
            return logits

        # CHANGED — use ImprovedTrainer with preprocess_logits_for_metrics
        trainer = ImprovedTrainer(
            class_weights=self.class_weights,
            focal_gamma=FOCAL_GAMMA,
            supcon_weight=SUPCON_WEIGHT,
            mixup_alpha=MIXUP_ALPHA,
            model=self.model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
            tokenizer=self.tokenizer,
            data_collator=data_collator,
            compute_metrics=self.compute_metrics,
            preprocess_logits_for_metrics=_preprocess_logits_for_metrics,  # CHANGED — fix inhomogeneous shape
            callbacks=[EarlyStoppingCallback(
                early_stopping_patience=EARLY_STOPPING_PATIENCE
            )],
        )

        print(f"Model: {self.model_name}")
        print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")
        print(f"Batch: {batch_size}x{GRAD_ACCUM_STEPS}, LR: {learning_rate}, MaxLen: {self.max_length}")
        print(f"Epochs: {num_epochs}, Eval every {eval_save_steps} steps")
        print(f"fp16: True, Scheduler: cosine, LLRD decay: {LLRD_DECAY}")
        print(f"Loss: FocalLoss(gamma={FOCAL_GAMMA}) + SupCon(w={SUPCON_WEIGHT})")
        print(f"Class weights: balanced (clipped@{MAX_CLASS_WEIGHT}), Label smoothing: {LABEL_SMOOTHING}")
        print()

        trainer.train()

        trainer.save_model()
        self.tokenizer.save_pretrained(output_dir)
        print(f"\n[{self.tag}] Training completed. Model saved to {output_dir}")
        return trainer

    # ── Evaluation ─────────────────────────────────────────
    def evaluate_model(self, trainer, val_dataset):  # CHANGED — add confusion matrix
        print(f"\n{'='*60}")
        print(f"  EVALUATION — {self.tag}")
        print(f"{'='*60}")

        predictions = trainer.predict(val_dataset)
        y_pred = np.argmax(predictions.predictions, axis=1)
        y_true = predictions.label_ids

        unique, counts = np.unique(y_pred, return_counts=True)
        print("\nPrediction distribution:")
        for u, c in zip(unique, counts):
            name = ID_TO_LABEL.get(int(u), f"unknown-{u}")
            print(f"  Predicted {name:>12s} (label {u}): {c} times")

        target_names = [ID_TO_LABEL[i] for i in range(NUM_LABELS)]
        print("\nPer-Class Classification Report:")
        print(classification_report(
            y_true, y_pred,
            target_names=target_names,
            zero_division=0,
        ))

        # CHANGED — confusion matrix
        print("Confusion Matrix:")
        cm = confusion_matrix(y_true, y_pred, labels=range(NUM_LABELS))
        cm_df = pd.DataFrame(cm, index=target_names, columns=target_names)
        print(cm_df)

        macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
        print(f"\n>>> COMPETITION METRIC (Macro F1): {macro_f1:.4f} <<<")
        return predictions

    # ── Full pipeline ──────────────────────────────────────
    def run_full_pipeline(
        self,
        output_dir=OUTPUT_DIR,
        num_epochs=NUM_EPOCHS,
        batch_size=BATCH_SIZE,
        learning_rate=LEARNING_RATE,
    ):
        try:
            train_df, val_df = self.load_and_prepare_data()
            self.initialize_model_and_tokenizer()
            train_dataset, val_dataset = self.prepare_datasets(train_df, val_df)
            trainer = self.train(
                train_dataset, val_dataset,
                output_dir=output_dir,
                num_epochs=num_epochs,
                batch_size=batch_size,
                learning_rate=learning_rate,
            )
            self.evaluate_model(trainer, val_dataset)
            print(f"\n[{self.tag}] Pipeline completed successfully!")
            return trainer, val_dataset
        except Exception as e:
            print(f"[{self.tag}] Error in pipeline: {e}")
            raise

print("GraphCodeBERTTrainerB defined.")


GraphCodeBERTTrainerB defined.


---
## Run Training + Evaluation

In [7]:
# ============================================================
# Cell 7: Run GraphCodeBERT on Task B
# ============================================================

print("\n" + "=" * 70)
print("  GraphCodeBERT — Task B (11-Class Authorship Detection) [IMPROVED]")
print("=" * 70)

trainer_obj = GraphCodeBERTTrainerB(
    model_name=MODEL_NAME,
    max_length=MAX_LENGTH,
)

hf_trainer, val_dataset = trainer_obj.run_full_pipeline(  # CHANGED — also return val_dataset
    output_dir=OUTPUT_DIR,
    num_epochs=NUM_EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
)



  GraphCodeBERT — Task B (11-Class Authorship Detection) [IMPROVED]
[graphcodebert-base] Dataset columns: ['code', 'generator', 'label', 'language']
[graphcodebert-base] Original dataset size: 500000

[graphcodebert-base] Unique labels: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10)]

[graphcodebert-base] Full dataset label distribution:
  0 (       human):  442096 (88.4%)
  1 (    deepseek):    4162 (0.8%)
  2 (        qwen):    8993 (1.8%)
  3 (       01-ai):    3029 (0.6%)
  4 (     bigcode):    2227 (0.4%)
  5 (       gemma):    1968 (0.4%)
  6 (         phi):    5783 (1.2%)
  7 (  meta-llama):    8197 (1.6%)
  8 ( ibm-granite):    8127 (1.6%)
  9 (     mistral):    4608 (0.9%)
  10 (      openai):   10810 (2.2%)

--- SUBSET MODE (target: 10% of dataset) ---
  Subsampled: 500000 -> 102904 samples (20.6%)
  Human: 45000 | Minority (all 10 classes): 57904

[graphcodebert-base] Class weig

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

...

Map:   0%|          | 0/10001 [00:00<?, ? examples/s]

[graphcodebert-base] Done. Train: 102904, Val: 10001

  STARTING TRAINING — graphcodebert-base
Model: microsoft/graphcodebert-base
Train: 102904, Val: 10001
Batch: 8x4, LR: 3e-05, MaxLen: 512
Epochs: 6, Eval every 1071 steps
fp16: True, Scheduler: cosine, LLRD decay: 0.9
Loss: FocalLoss(gamma=2.0) + SupCon(w=0.2)
Class weights: balanced (clipped@8.0), Label smoothing: 0.0



Step,Training Loss,Validation Loss,Macro F1,Weighted F1,Accuracy
1071,1.240800,1.041713,0.348790,0.876987,0.852915
2142,1.084700,0.992421,0.399784,0.887678,0.869013
3213,1.015800,1.002367,0.415289,0.889037,0.866413
4284,0.943900,0.946794,0.470721,0.908371,0.894111
5355,0.794400,0.973968,0.455676,0.900080,0.883612
6426,0.840500,0.938908,0.468137,0.909260,0.896510


  Prediction distribution: {0: 8053, 1: 393, 2: 102, 3: 94, 4: 160, 5: 178, 6: 169, 7: 103, 8: 375, 9: 64, 10: 310}
  Per-class F1: hum=0.95 | dee=0.15 | qwe=0.17 | 01-=0.21 | big=0.30 | gem=0.21 | phi=0.49 | met=0.18 | ibm=0.45 | mis=0.12 | ope=0.59
  Prediction distribution: {0: 8148, 1: 366, 2: 38, 3: 86, 4: 113, 5: 149, 6: 111, 7: 63, 8: 317, 9: 198, 10: 412}
  Per-class F1: hum=0.96 | dee=0.20 | qwe=0.14 | 01-=0.29 | big=0.45 | gem=0.25 | phi=0.53 | met=0.27 | ibm=0.52 | mis=0.19 | ope=0.62
  Prediction distribution: {0: 8081, 1: 292, 2: 162, 3: 121, 4: 166, 5: 155, 6: 109, 7: 125, 8: 314, 9: 164, 10: 312}
  Per-class F1: hum=0.95 | dee=0.25 | qwe=0.33 | 01-=0.24 | big=0.37 | gem=0.28 | phi=0.55 | met=0.27 | ibm=0.50 | mis=0.17 | ope=0.65
  Prediction distribution: {0: 8353, 1: 198, 2: 157, 3: 177, 4: 131, 5: 63, 6: 114, 7: 167, 8: 196, 9: 179, 10: 266}
  Per-class F1: hum=0.97 | dee=0.35 | qwe=0.27 | 01-=0.26 | big=0.42 | gem=0.44 | phi=0.57 | met=0.33 | ibm=0.61 | mis=0.30 | ope

  Prediction distribution: {0: 8353, 1: 198, 2: 157, 3: 177, 4: 131, 5: 63, 6: 114, 7: 167, 8: 196, 9: 179, 10: 266}
  Per-class F1: hum=0.97 | dee=0.35 | qwe=0.27 | 01-=0.26 | big=0.42 | gem=0.44 | phi=0.57 | met=0.33 | ibm=0.61 | mis=0.30 | ope=0.67

Prediction distribution:
  Predicted        human (label 0): 8353 times
  Predicted     deepseek (label 1): 198 times
  Predicted         qwen (label 2): 157 times
  Predicted        01-ai (label 3): 177 times
  Predicted      bigcode (label 4): 131 times
  Predicted        gemma (label 5): 63 times
  Predicted          phi (label 6): 114 times
  Predicted   meta-llama (label 7): 167 times
  Predicted  ibm-granite (label 8): 196 times
  Predicted      mistral (label 9): 179 times
  Predicted       openai (label 10): 266 times

Per-Class Classification Report:
              precision    recall  f1-score   support

       human       1.00      0.94      0.97      8849
    deepseek       0.25      0.58      0.35        85
        qwen      

## Per-Class Threshold Calibration  # CHANGED — entirely new

Sweep per-class decision thresholds on the validation set to maximise macro F1.

In [8]:
# ============================================================
# Cell 8: Per-Class Threshold Calibration  # CHANGED — entirely new
# ============================================================

def calibrate_thresholds(logits, y_true, num_labels=NUM_LABELS):
    """Find per-class thresholds that maximise macro F1."""
    from scipy.special import softmax as sp_softmax
    probs = sp_softmax(logits, axis=1)
    thresholds = np.zeros(num_labels)

    for cls_idx in range(num_labels):
        binary_true = (y_true == cls_idx).astype(int)
        cls_probs = probs[:, cls_idx]

        best_f1, best_thresh = 0, 0.5
        for t in np.arange(0.05, 0.95, 0.02):
            preds = (cls_probs >= t).astype(int)
            if preds.sum() == 0:
                continue
            f1 = f1_score(binary_true, preds, zero_division=0)
            if f1 > best_f1:
                best_f1, best_thresh = f1, t
        thresholds[cls_idx] = best_thresh
        name = ID_TO_LABEL.get(cls_idx, f"cls-{cls_idx}")
        print(f"  {name:>12s}: threshold={best_thresh:.2f}, binary_f1={best_f1:.3f}")

    return thresholds


def predict_with_thresholds(logits, thresholds):
    """Apply calibrated thresholds; fallback to argmax."""
    from scipy.special import softmax as sp_softmax
    probs = sp_softmax(logits, axis=1)
    preds = np.full(len(logits), -1, dtype=int)

    for i in range(len(logits)):
        above = np.where(probs[i] >= thresholds)[0]
        if len(above) > 0:
            preds[i] = above[np.argmax(probs[i][above])]
        else:
            preds[i] = np.argmax(probs[i])
    return preds


# --- Run calibration ---
print("\n--- Per-Class Threshold Calibration ---")
val_predictions = hf_trainer.predict(val_dataset)
val_logits = val_predictions.predictions
val_labels = val_predictions.label_ids

thresholds = calibrate_thresholds(val_logits, val_labels)
print(f"\nCalibrated thresholds: {thresholds}")

# --- Evaluate with calibrated thresholds ---
cal_preds = predict_with_thresholds(val_logits, thresholds)
cal_macro_f1 = f1_score(val_labels, cal_preds, average='macro', zero_division=0)

target_names = [ID_TO_LABEL[i] for i in range(NUM_LABELS)]
print("\nCalibrated Per-Class Report:")
print(classification_report(
    val_labels, cal_preds,
    target_names=target_names,
    zero_division=0,
))

# CHANGED — confusion matrix with calibrated thresholds
print("\nCalibrated Confusion Matrix:")
cm = confusion_matrix(val_labels, cal_preds, labels=range(NUM_LABELS))
print(pd.DataFrame(cm, index=target_names, columns=target_names))

print(f"\n>>> CALIBRATED Macro F1: {cal_macro_f1:.4f} <<<")

# Compare
uncal_preds = np.argmax(val_logits, axis=1)
uncal_f1 = f1_score(val_labels, uncal_preds, average='macro', zero_division=0)
print(f">>> UNCALIBRATED Macro F1: {uncal_f1:.4f} <<<")
print(f">>> IMPROVEMENT: +{(cal_macro_f1 - uncal_f1)*100:.2f} pts <<<")



--- Per-Class Threshold Calibration ---


  Prediction distribution: {0: 8353, 1: 198, 2: 157, 3: 177, 4: 131, 5: 63, 6: 114, 7: 167, 8: 196, 9: 179, 10: 266}
  Per-class F1: hum=0.97 | dee=0.35 | qwe=0.27 | 01-=0.26 | big=0.42 | gem=0.44 | phi=0.57 | met=0.33 | ibm=0.61 | mis=0.30 | ope=0.67
         human: threshold=0.05, binary_f1=0.987
      deepseek: threshold=0.53, binary_f1=0.451
          qwen: threshold=0.27, binary_f1=0.367
         01-ai: threshold=0.49, binary_f1=0.310
       bigcode: threshold=0.45, binary_f1=0.547
         gemma: threshold=0.55, binary_f1=0.607
           phi: threshold=0.29, binary_f1=0.583
    meta-llama: threshold=0.31, binary_f1=0.413
   ibm-granite: threshold=0.33, binary_f1=0.628
       mistral: threshold=0.31, binary_f1=0.362
        openai: threshold=0.27, binary_f1=0.713

Calibrated thresholds: [0.05 0.53 0.27 0.49 0.45 0.55 0.29 0.31 0.33 0.31 0.27]

Calibrated Per-Class Report:
              precision    recall  f1-score   support

       human       0.99      0.97      0.98      8849


---
## Predict on Test Set

In [9]:
# ============================================================
# Cell 9: Test set prediction  # CHANGED — uses calibrated thresholds
# ============================================================
from itertools import chain
from tqdm import tqdm


@torch.no_grad()
def predict_with_trainer(trainer_obj, parquet_path, output_path,
                         max_length=512, batch_size=16, device=None,
                         thresholds=None):  # CHANGED — accept thresholds
    """Run inference with optional threshold calibration."""
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    model = trainer_obj.model
    tokenizer = trainer_obj.tokenizer
    if tokenizer is None:
        raise ValueError("trainer_obj must have a tokenizer.")

    model.to(device)
    model.eval()

    from datasets import load_dataset
    ds = load_dataset("parquet", data_files=parquet_path, split="train", streaming=True)

    it = iter(ds)
    first = next(it)
    id_col = "ID" if "ID" in first else "id" if "id" in first else None
    if "code" not in first:
        raise ValueError("Parquet must contain a 'code' column")
    stream = chain([first], it)

    def batcher(iterator, bs):
        buf = []
        for ex in iterator:
            buf.append(ex)
            if len(buf) == bs:
                yield buf
                buf = []
        if buf:
            yield buf

    with open(output_path, "w") as f:
        f.write("ID,prediction\n")
        global_idx = 0
        for batch in tqdm(batcher(stream, batch_size), desc="Predicting"):
            codes = [row["code"] for row in batch]
            if id_col:
                ids = [row[id_col] for row in batch]
            else:
                ids = list(range(global_idx, global_idx + len(batch)))
                global_idx += len(batch)
            enc = tokenizer(
                codes, truncation=True, padding=True,
                max_length=max_length, return_tensors="pt",
            )
            input_ids = enc["input_ids"].to(device)
            attention_mask = enc["attention_mask"].to(device)
            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
            logits_np = logits.cpu().numpy()

            # CHANGED — use calibrated thresholds if available
            if thresholds is not None:
                pred_labels = predict_with_thresholds(logits_np, thresholds).tolist()
            else:
                pred_labels = logits.argmax(dim=-1).cpu().tolist()

            for ex_id, pred in zip(ids, pred_labels):
                f.write(f"{ex_id},{pred}\n")

    print(f"Predictions saved to {output_path}")

print("predict_with_trainer() defined.")


predict_with_trainer() defined.


In [10]:
# ============================================================
# Cell 10: Generate submission  # CHANGED — pass thresholds
# ============================================================

if TEST_PATH is not None:
    OUT_CSV = "submission_b_graphcodebert.csv"
    predict_with_trainer(
        trainer_obj=trainer_obj,
        parquet_path=TEST_PATH,
        output_path=OUT_CSV,
        max_length=MAX_LENGTH,
        batch_size=32,
        device="cuda",
        thresholds=thresholds,  # CHANGED — use calibrated thresholds
    )
    print("Wrote:", OUT_CSV)
else:
    print("No test path set. Update TEST_PATH in the data setup cell.")


Predicting: 32it [00:36,  1.15s/it]

Predictions saved to submission_b_graphcodebert.csv
Wrote: submission_b_graphcodebert.csv


In [11]:
# ============================================================
# Cell 11: Clean up GPU memory
# ============================================================

del hf_trainer, trainer_obj
gc.collect()
torch.cuda.empty_cache()
print("GPU memory freed.")


GPU memory freed.
